In [1]:
import pyNUISANCE as pn

In [5]:
input_vectors = {
    -14: pn.EventSource("/root/Projects/pico/NOEC/evs/NEUT.dune_ND_numu.gen_numubar.hepmc3.gz"),
    -12: pn.EventSource("/root/Projects/pico/NOEC/evs/NEUT.dune_ND_numu.gen_nuebar.hepmc3.gz"),
    12: pn.EventSource("/root/Projects/pico/NOEC/evs/NEUT.dune_ND_numu.gen_nue.hepmc3.gz"),
    14: pn.EventSource("/root/Projects/pico/NOEC/evs/NEUT.dune_ND_numu.gen_numu.hepmc3.gz")
}

for nupdg, evs in input_vectors.items():
    if not evs:
        raise RuntimeError(f"Failed to read event file for {nupdg}")

In [6]:
from pyProSelecta import event, part, unit, pdg, p3mod, momentum, kinetic_energy, ext
from pyNUISANCE import nhm
import numpy as np

def GetPrimaryLepton(ev):
    beam_part = nhm.EventUtils.GetBeamParticle(ev)
    prim_vertex = nhm.EventUtils.GetPrimaryVertex(ev)
    for part in prim_vertex.particles_out():
        if (part.pid() == beam_part.pid()) or\
           ( (abs(part.pid()) + 1 ) == abs(beam_part.pid())):
          return part
    raise runtime_error("Failed to find primary lepton out: %s -> [%s]" % \
                        (beam_part.pid(), ",".join(part.pid for part in prim_vertex.particles_out())))

def PrimaryLeptonTotalEnergy(ev):
    pl = GetPrimaryLepton(ev)
    return (pl.momentum().e() / unit.GeV_c) if pl.pid() % 2 else 0

def NeutralPionTotalEnergy(ev):
    return part.sum(momentum, event.all_out_part(ev, 111)).e() / unit.GeV_c

def NumberOfChargedPions(ev):
    return event.num_out_part(ev, [211,-211], flatten=True)

def ChargedPionKineticEnergy(ev):
    return part.sum(kinetic_energy, event.all_out_part(ev, [211, -211], flatten=True)) / unit.GeV_c

def ProtonKineticEnergy(ev):
    return part.sum(kinetic_energy, event.all_out_part(ev, 2212)) / unit.GeV_c

def ReconstructedNeutrinoEnergy_true(ev):
    return PrimaryLeptonTotalEnergy(ev) + ChargedPionKineticEnergy(ev) + (NumberOfChargedPions(ev)*0.1395) +  NeutralPionTotalEnergy(ev) + ProtonKineticEnergy(ev)

event_frames = {}

for nupdg, evs in input_vectors.items():
    event_frames[nupdg] = pn.EventFrameGen(evs) \
                            .add_column("pid_nu", lambda ev: nhm.EventUtils.GetBeamParticle(ev).pid()) \
                            .add_column("E_nu", ext.enu_GeV) \
                            .add_column("is_CC", ext.isCC) \
                            .add_column("pid_lep", lambda ev: GetPrimaryLepton(ev).pid()) \
                            .add_column("E_lepton", PrimaryLeptonTotalEnergy) \
                            .add_column("T_proton", ProtonKineticEnergy) \
                            .add_column("num_pi0", lambda ev: event.num_out_part(ev, 111)) \
                            .add_column("E_pi0", NeutralPionTotalEnergy) \
                            .add_column("num_cpi", NumberOfChargedPions) \
                            .add_column("T_cpi", ChargedPionKineticEnergy) \
                            .add_column("E_nu_rec_true", ReconstructedNeutrinoEnergy_true).all()

output_columns = [
    "pid_nu",
    "E_nu",
    "is_CC",
    "pid_lep",
    "E_lepton",
    "T_proton",
    "num_pi0",
    "E_pi0",
    "num_cpi",
    "T_cpi",
    "E_nu_rec_true" ]

print(event_frames[14])

 --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 | event.number | weight.cv | fatx_per_su$ | fatx_per_su$ | process.id | pid_nu |   E_nu | is_CC | pid_lep | E_lepton | T_proton | num_pi0 | E_pi0 | num_cpi |   T_cpi | E_nu_rec_tr$ |
 --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 |            0 |         1 |    1.374e-06 |    3.434e-08 |        500 |     14 |  3.222 |     1 |      13 |    2.019 |   0.3915 |       0 |     0 |       2 |  0.5405 |         3.23 |
 |            1 |         1 |    6.869e-07 |    1.717e-08 |        251 |     14 |    3.2 |     0 |      14 |        0 |        0 |       0 |     0 |       0 |       0 |            0 |
 |            2 |         1 |    4.579e-07 |    1.145e-08 |        200 |     14 

In [7]:
import pandas as pa

def todf(ef, index):
  df = pa.DataFrame(data=ef.table,
                    index=index,
                    columns=ef.column_names)
  for row in ["event.number", "process.id", "pid_nu", "is_CC", "pid_lep", "num_pi0", "num_cpi"]:
    df[row] = pa.to_numeric(df[row], downcast="integer")
  return df

In [9]:
df_numu = todf(event_frames[14], [i for i in range(event_frames[14].table.shape[0])])
df_nue = todf(event_frames[12], [i + event_frames[14].table.shape[0] for i in range(event_frames[12].table.shape[0])])
df_numode = pa.concat([df_numu, df_nue])
df_numode = df_numode.sample(frac=1).reset_index(drop=True)
df_numode

,event.number,weight.cv,fatx_per_sumw.pb_per_target.estimate,fatx_per_sumw.pb_per_nucleon.estimate,process.id,pid_nu,E_nu,is_CC,pid_lep,E_lepton,T_proton,num_pi0,E_pi0,num_cpi,T_cpi,E_nu_rec_true
0,63321,1.0,2.208073e-11,5.520182e-13,251,12,0.990967,0,12,0.000000,0.052395,0,0.000000,0,0.000000,0.052395
1,51275,1.0,2.679137e-11,6.697844e-13,600,14,3.982955,1,13,1.003662,2.065108,0,0.000000,2,0.406640,3.754410
2,30557,1.0,4.575547e-11,1.143887e-12,200,12,1.090421,1,11,1.027498,0.048758,0,0.000000,0,0.000000,1.076257
3,56087,1.0,2.492861e-11,6.232152e-13,600,12,3.425913,1,11,1.196185,1.739150,1,0.510285,0,0.000000,3.445619
4,15998,1.0,8.586502e-11,2.146626e-12,452,14,2.660112,0,14,0.000000,0.291461,0,0.000000,1,0.424332,0.855293
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,19350,1.0,7.099140e-11,1.774785e-12,550,14,1.830639,0,14,0.000000,0.505998,0,0.000000,1,0.259189,0.904687
199996,24314,1.0,5.649823e-11,1.412456e-12,500,14,1.788368,1,13,0.651629,0.000000,0,0.000000,2,0.503424,1.434054
199997,7423,1.0,1.850424e-10,4.626059e-12,600,14,20.888543,1,13,17.101732,0.209833,0,0.000000,2,2.400774,19.991340
199998,17141,1.0,8.013969e-11,2.003492e-12,401,14,2.208387,1,13,0.901962,1.069667,0,0.000000,0,0.000000,1.971628


In [10]:
df_numode.to_csv("neutrino_mode_events.csv",columns=output_columns, float_format="%.4f")

In [11]:
df_numubar = todf(event_frames[-14], [i for i in range(event_frames[-14].table.shape[0])])
df_nuebar = todf(event_frames[-12], [i + event_frames[-14].table.shape[0] for i in range(event_frames[12].table.shape[0])])
df_nubarmode = pa.concat([df_numubar, df_nuebar])
df_nubarmode = df_nubarmode.sample(frac=1).reset_index(drop=True)
df_nubarmode

,event.number,weight.cv,fatx_per_sumw.pb_per_target.estimate,fatx_per_sumw.pb_per_nucleon.estimate,process.id,pid_nu,E_nu,is_CC,pid_lep,E_lepton,T_proton,num_pi0,E_pi0,num_cpi,T_cpi,E_nu_rec_true
0,59662,1.0,8.963123e-12,2.240781e-13,525,-14,3.821926,1,-13,2.642871,0.067250,2,0.517747,0,0.000000,3.227868
1,37139,1.0,1.464502e-11,3.661254e-13,425,-12,2.706585,1,-11,1.919416,0.061314,0,0.000000,0,0.000000,1.980730
2,4253,1.0,1.257092e-10,3.142729e-12,425,-14,6.451917,1,-13,5.457074,0.000000,0,0.000000,1,0.034058,5.630632
3,46908,1.0,1.159513e-11,2.898782e-13,225,-12,0.665801,1,-11,0.601541,0.000000,0,0.000000,0,0.000000,0.601541
4,14005,1.0,3.818127e-11,9.545317e-13,425,-14,2.526979,1,-13,1.475540,0.036516,0,0.000000,1,0.353351,2.004907
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,58303,1.0,9.328964e-12,2.332241e-13,575,-12,1.744510,0,-12,0.000000,0.532956,0,0.000000,1,0.049993,0.722449
199996,95023,1.0,5.723985e-12,1.430996e-13,225,-12,2.921047,1,-11,2.813289,0.000000,0,0.000000,0,0.000000,2.813289
199997,47458,1.0,1.146075e-11,2.865188e-13,425,-12,4.586149,1,-11,4.202133,0.000000,0,0.000000,1,0.082067,4.423700
199998,9134,1.0,5.954197e-11,1.488549e-12,225,-12,2.543986,1,-11,2.341150,0.117642,0,0.000000,0,0.000000,2.458793
